# Project — Will the client subscribe? A cost-aware bank-marketing classifier

A bank runs a phone campaign selling **term deposits**. Calling everyone is expensive, and
only about **11%** of clients subscribe. Your job is the same shape as the cheese factory from
the lectures: turn a model's **scores** into a **decision** that respects an **asymmetric cost**
(a wasted call is cheap; a missed subscriber is expensive), and report **probabilities you can
trust**.

You will build the whole pipeline end to end:

1. **Setup & first look** — load the data, see the imbalance.
2. **Split & a baseline to beat** — train/val/test; why "predict no one" is useless despite 88% accuracy.
3. **The leakage trap** — one feature (`duration`) inflates the score; measure it, then drop it.
4. **Logistic regression & interpretation** — fit it, read the coefficients as odds ratios.
5. **The right metric & operating point** — PR over ROC on imbalanced data; tune the threshold **by cost**.
6. **Calibration** — are the probabilities trustworthy? (and what `class_weight` does to them, and how to fix it.)
7. **Sealed-test evaluation & model card** — score once, write it up.
8. **Imbalanced learning** — do class weights / SMOTE actually beat the threshold?

Dataset: UCI Bank Marketing (`data/bank-full.csv`), 45,211 clients, target `y` = subscribed (yes/no).

## Part 1 — Setup & first look

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.calibration import calibration_curve
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)

SEED = 509
ARM_BLUE, ARM_RED, ARM_ORANGE = "#0033A0", "#D90012", "#F2A800"

# The data is semicolon-separated. Load it and turn y into a 0/1 integer column.
df = pd.read_csv("data/bank-full.csv", sep=";")
df["y"] = (df["y"] == "yes").astype(int)
print("shape:", df.shape)
df.head()

**The target.** How (im)balanced is it? What accuracy would a model that just predicts the majority class get?

In [ ]:
# TODO: print the positive rate (mean of y) and the value_counts.
# Then compute what accuracy a "predict nobody subscribes" model would get.
rate = ...
print(...)

**Data quirks (look before you model).** Two things to handle:

- `pdays = -1` is a **sentinel** meaning "this client was never contacted in a previous campaign" — not a real "-1 days". Feeding it raw to a scaler is wrong. We add a `was_contacted_before` flag and set those `pdays` to 0.
- Several categorical columns contain an `"unknown"` level (e.g. `contact`, `poutcome`). That is a legitimate category — one-hot encoding will just give it its own column.

In [ ]:
# TODO: add a binary `was_contacted_before` flag from pdays != -1,
#       then set the sentinel pdays == -1 to 0.
df["was_contacted_before"] = ...
df.loc[..., "pdays"] = 0

Define the column groups we will preprocess differently. (`duration` is included **for now** — Part 3 deals with it.)

In [ ]:
# TODO: list the numeric and categorical feature names.
# Numeric: age, balance, day, duration, campaign, pdays, previous, was_contacted_before
# Categorical: job, marital, education, default, housing, loan, contact, month, poutcome
num_cols = [...]
cat_cols = [...]
X = df.drop(columns="y")
y = df["y"].values

## Part 2 — Split & a baseline to beat

We use **three** splits, not two:

- **train** — fit the model;
- **validation** — pick the threshold and check calibration;
- **test** — touched **once**, at the very end, for the number we report.

Tuning the threshold on the same data you report on would bias the result optimistically.

In [ ]:
# TODO: a stratified three-way split (60/20/20), random_state=SEED.
# Hint: split off 40% first, then halve that into val and test.
X_tr, X_tmp, y_tr, y_tmp = train_test_split(...)
X_val, X_te, y_val, y_te = train_test_split(...)
print(f"train={len(y_tr)}, val={len(y_val)}, test={len(y_te)}")

**The baseline.** `DummyClassifier` predicts the majority class. High accuracy, zero recall — the lecture's whole point, on real data.

In [ ]:
# TODO: fit a DummyClassifier(strategy="most_frequent") and report its
#       accuracy AND recall on the validation set. Note the contrast.
dummy = ...

## Part 3 — Preprocess, then the leakage trap

**Preprocessing in a `Pipeline`** so scaling/encoding are fit on *train only* (no leakage):
`StandardScaler` for numerics, `OneHotEncoder(drop="first")` for categoricals.

In [ ]:
# TODO: write a helper returning a Pipeline of
#   ColumnTransformer(StandardScaler on `numeric`, OneHotEncoder(drop="first") on cat_cols)
#   -> LogisticRegression(max_iter=2000, ...).
# Keep the extra args (penalty/C/solver) -- the optional L1 task at the end uses them.
def make_pipe(numeric, class_weight=None, penalty="l2", C=1.0, solver="lbfgs"):
    pre = ColumnTransformer([...])
    return Pipeline([...])

**The leakage trap.** `duration` is the length of the call in seconds. You only know it
*after* the call has happened — so a model that uses it to decide *whom to call* is cheating.
Let's measure how much it inflates the score, then drop it.

In [ ]:
# TODO: fit one model WITH duration and one WITHOUT, and compare val ROC AUC.
#   num_honest = [c for c in num_cols if c != "duration"]
# Keep the honest model as `pipe` and its val scores as `s_val` -- we use them below.
pipe_leaky = ...
pipe = ...
s_val = ...

> **Your takeaway (write 1–2 sentences).** Which AUC is the honest one, and why is the higher
> one misleading at decision time? Which model do you keep for the rest of the project?

## Part 4 — Logistic regression & interpreting it

`pipe` is our fitted logistic regression. Logreg's superpower is interpretability:
`exp(coef)` is an **odds ratio**. Because the numerics are standardized, their odds ratios are
**per one standard deviation** (handy: they're directly comparable). One-hot columns are 0/1,
so their odds ratio is "this category vs the dropped reference" — so print the reference too.

In [ ]:
# TODO: pull the fitted coefficients out of the pipeline and turn them into odds ratios.
#   ohe = pipe.named_steps["pre"].named_transformers_["cat"]
#   feat_names = num_honest + list(ohe.get_feature_names_out(cat_cols))
#   coef = pipe.named_steps["clf"].coef_.ravel()
# Rank exp(coef) and plot a horizontal bar chart of the most extreme features (line at 1).
# Also print the dropped reference category per feature so the categorical odds ratios are
# readable:  {c: list(cats)[0] for c, cats in zip(cat_cols, ohe.categories_)}

> **Caveat.** These are **standardized** (per-SD) odds ratios — great for comparing feature
> strength, but *not* "per extra euro / per extra year". For per-unit odds ratios you would fit
> on the raw, unscaled features.

## Part 5 — The right metric, then the operating point

At 11% positives, **accuracy and ROC overstate**. Report **PR AUC (average precision)** alongside
ROC, and read precision/recall at an operating point.

In [ ]:
# TODO: compute ROC AUC and PR AUC (average_precision_score) on validation,
#       and plot the ROC curve and PR curve side by side.
#       On the PR plot, draw the base-rate line (y_val.mean()).

**Choose the threshold by cost, not by 0.5.** Suppose a wasted call costs **€5** and a missed
subscriber costs **€100** (lost net value). We count only the **error** costs (a made sale nets
value minus the call); minimise expected error cost on validation.
Because we did **not** reweight the classes, the probabilities are honest, so the empirical
optimum should land near the closed-form `C_FP / (C_FN + C_FP)`.

In [ ]:
# TODO: define the cost (C_FP=5, C_FN=100), sweep the threshold over [0.01, 0.99] on validation,
#       find the cost-minimising c*, plot cost vs threshold, and compare c* to the closed-form
#       C_FP / (C_FN + C_FP). Keep c_star for the final evaluation.
C_FP, C_FN = 5, 100
...
c_star = ...

## Part 6 — Are the probabilities trustworthy? (calibration)

A cost decision (and any "this client has an 80% chance" statement) only means something if the
probabilities are **calibrated**. Logistic regression minimises log-loss (a proper scoring rule),
so it is usually well calibrated — let's check, and then see what `class_weight="balanced"` does to it.

In [ ]:
# TODO: write an ece() helper, then draw a reliability diagram comparing the
#       no-weight logreg scores (s_val) with a class_weight="balanced" version.
#       Report ECE for each. Which one tracks the diagonal? What did reweighting do
#       to the average predicted probability vs the true 11% base rate?

> **Your takeaway (write 1–2 sentences).** Which curve tracks the diagonal? What did
> `class_weight="balanced"` do to the average predicted probability vs the 11% base rate, and
> what would that force you to do? Why did we handle the imbalance with the *threshold* instead?

**Fixing a decalibrated model — the tool for when you *do* reweight.** We rejected `class_weight` because the plain logreg was already calibrated. But many models aren't (a reweighted logreg, and later boosted trees or SVMs). Deck [14]'s answer: don't retrain — learn a small post-hoc **isotonic** map from the raw scores to true frequencies with `CalibratedClassifierCV`, fit by CV on held-out data. See if it undoes the damage `class_weight` did.

In [ ]:
# TODO: recalibrate the class_weight="balanced" model with isotonic regression.
#   from sklearn.calibration import CalibratedClassifierCV
#   cal_bal = CalibratedClassifierCV(make_pipe(num_honest, class_weight="balanced"),
#                                    method="isotonic", cv=5).fit(X_tr, y_tr)
#   s_val_bal_cal = cal_bal.predict_proba(X_val)[:, 1]
# Then plot balanced-raw (s_val_bal) vs balanced+isotonic on one reliability diagram, and
# report ECE and the mean predicted probability for each vs the true rate (y_val.mean()).
cal_bal = ...
s_val_bal_cal = ...

> **Your takeaway (1 sentence).** What did isotonic do to the ECE and to the mean predicted probability, and when would you reach for it?

## Part 7 — Sealed-test evaluation & model card

Now, and only now, touch the test set: apply the honest pipeline at `c*` **once**.
(For an actual deployment you would refit on train+val before shipping; we keep the train-only
model so this test estimate stays honest.)

In [ ]:
# TODO: score the test set with `pipe`, threshold at c*, and report PR AUC, recall,
#       precision, F1, and the total cost vs the default-0.5 cost. Plot the confusion matrix.
#       This is the ONLY time you touch the test set.

### Model card (fill this in)

Summarise *your* model. Keep it to one line per field.

| Field | Value |
|---|---|
| **Task** | ... |
| **Data** | ... |
| **Leakage handled** | ... |
| **Model** | ... |
| **Headline metric** | ... |
| **Operating point** | ... |
| **Calibration** | ... |
| **Limitations** | ... |

> **One-line takeaway:** ...

## Part 8 — Would reweighting or resampling have helped?

We handled the 11% imbalance with the **metric** (PR AUC) and the **cost threshold** — never touching the classes. Deck [15] offers the heavier tools: **class weights** and **resampling** (`RandomOverSampler`, and **SMOTENC** for our 9 categorical columns). Do they actually beat the baseline?

The honest test is **PR AUC in 5-fold CV**, with every resampler fit **inside each fold** (`imblearn.Pipeline`) — resampling before the split leaks synthetic points across train/test and gives a fantasy score.

In [ ]:
from imblearn.over_sampling import SMOTENC, RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import cross_val_score

# TODO: build four pipelines and compare 5-fold CV PR-AUC on the TRAINING set.
#   - "baseline" and "class_weight" (="balanced"): plain sklearn Pipeline(pre -> logreg).
#   - "ROS" (RandomOverSampler) and "SMOTENC": imblearn Pipeline with the sampler as the FIRST
#     step, so it resamples inside each CV fold -- never before the split (leakage!).
#   SMOTENC needs categorical column POSITIONS:  cat_idx = [X_tr.columns.get_loc(c) for c in cat_cols]
#   Build the preprocessor with OneHotEncoder(drop="first", handle_unknown="ignore")
#   (a rare category can be missing from a fold). Score each with
#   cross_val_score(pipe, X_tr, y_tr, scoring="average_precision", cv=5).
# Plot the four PR-AUCs as a bar chart (with a baseline reference line) and print mean +/- std.
approaches = {...}

In [ ]:
# TODO: fit each approach on train, predict on validation, and print val PR-AUC, ECE, and the
#       mean predicted probability. Which methods push the mean prediction off the 11% base rate
#       (i.e. decalibrate), and does any of them actually improve PR AUC?

> **Your takeaway (2–3 sentences).** Did any resampler beat the baseline's PR AUC? What did they do to ECE and the mean prediction? Given that, was handling the imbalance with the *threshold* (Part 5) the right call — and when *would* resampling actually help?

## Going further (optional)

**A. Lift / gain — "we can only call the top 20%."** How many subscribers do we catch?
(Use the **validation** set — the test set is already spent.)

In [ ]:
# TODO (optional): rank the VALIDATION set by score, plot the cumulative-gain curve, and read
#       off what fraction of subscribers you catch in the top 20% (and the lift vs random).

**B. Sparsity with L1.** Does an L1 penalty zero out features without hurting PR AUC?

In [ ]:
# TODO (optional): fit an L1-penalised logreg via make_pipe(num_honest, penalty="l1",
#       solver="liblinear", C=0.1), count how many coefficients went to zero, and compare its
#       val PR AUC to the full model.